In [42]:
import json
import re
from pathlib import Path

import fitz  # PyMuPDF

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
pdf_path = project_root / "verifikacija_softvera.pdf"
pages_dir = out_dir / "pages" 
pages_dir.mkdir(parents=True, exist_ok=True)

FIRST_N_PAGES = 204
print(pdf_path, pdf_path.exists())

/home/anja/Desktop/MU/Student-Question-Answering-from-Course-Materials/verifikacija_softvera.pdf True


In [43]:
pip install pymupdf 

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [44]:
import sys
!{sys.executable} -m pip install "PyMuPDF==1.24.14"


Defaulting to user installation because normal site-packages is not writeable
ERROR: Ignored the following yanked versions: 1.18.11
ERROR: Could not find a version that satisfies the requirement PyMuPDF==1.24.14 (from versions: 1.11.2, 1.12.5, 1.13.20, 1.14.19.post2, 1.14.20, 1.14.21, 1.16.0, 1.16.1, 1.16.2, 1.16.3, 1.16.4, 1.16.5, 1.16.6, 1.16.7, 1.16.8, 1.16.9, 1.16.10, 1.16.11, 1.16.12, 1.16.13, 1.16.14, 1.16.15, 1.16.16, 1.16.17, 1.16.18, 1.17.0, 1.17.1, 1.17.2, 1.17.3, 1.17.4, 1.17.5, 1.17.6, 1.17.7, 1.18.0, 1.18.1, 1.18.2, 1.18.3, 1.18.4, 1.18.5, 1.18.6, 1.18.7, 1.18.8, 1.18.9, 1.18.10, 1.18.12, 1.18.13, 1.18.14, 1.18.15, 1.18.16, 1.18.17, 1.18.18, 1.18.19, 1.19.0, 1.19.1, 1.19.2, 1.19.3, 1.19.4, 1.19.5, 1.19.6, 1.20.0, 1.20.1, 1.20.2, 1.21.0, 1.21.1, 1.22.0, 1.22.1, 1.22.2, 1.22.3, 1.22.5, 1.23.0rc1, 1.23.0rc2, 1.23.0, 1.23.1, 1.23.2rc1, 1.23.2, 1.23.3, 1.23.4, 1.23.5, 1.23.6, 1.23.7, 1.23.8, 1.23.9rc1, 1.23.9rc2, 1.23.9, 1.23.10, 1.23.11, 1.23.12, 1.23.13, 1.23.14, 1.23.15, 1.2

In [45]:
WATERMARK_TEXT = "Elektronska verzĳa (2026)"
WATERMARK_SIZE_THRESHOLD = 30.0
HEADING_SIZE_THRESHOLD = 12.0
HEADER_BAND_Y = 45.0

NUMBERED_HEADING_RE = re.compile(r"^(\d+(?:\.\d+)*)\s+(.*\S)\s*$")
PAGE_NUM_RE = re.compile(r"\b(\d{1,4})\b")
CONTROL_CHARS_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

In [46]:
from typing import Optional, Tuple


def _block_text(block: dict) -> str:

    lines = []
    for line in block.get("lines", []):
        spans = "".join(s["text"] for s in line["spans"])
        lines.append(spans)

    text = "\n".join(lines).strip()
    text = CONTROL_CHARS_RE.sub("", text)

    #Strelice
    text = text.replace("▶", "-")

    #Prelomljene reči
    text = re.sub(r"(?<=\w)-\s*\n\s*(?=\w)", "", text)

    #Ostali novi redovi
    text = text.replace("\n", " ")

    #Višestruki razmaci
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def _block_max_size(block: dict) -> float:
    sizes = [s["size"] for line in block.get("lines", []) for s in line["spans"]]
    return max(sizes) if sizes else 0.0


def _heading_level(text: str) -> Optional[Tuple[int, str]]:
    match = NUMBERED_HEADING_RE.match(text.replace("\n", " "))
    if not match:
        return None
    number, title = match.groups()
    level = number.count(".") + 1
    return level, f"{number} {title}"

In [47]:
def extract_page(page: fitz.Page) -> dict:
    raw = page.get_text("dict")
    body_blocks: list[tuple[float, float, str, float]] = []
    header_text = None

    for block in raw.get("blocks", []):
        if block.get("type", 0) != 0:
            continue  # preskacemo slike
        text = _block_text(block)
        if not text:
            continue
        max_size = _block_max_size(block)
        x0, y0 = block["bbox"][0], block["bbox"][1]

        if text.startswith(WATERMARK_TEXT) or max_size >= WATERMARK_SIZE_THRESHOLD:
            continue  #onaj znak na strani

        if y0 < HEADER_BAND_Y:
            header_text = text.replace("\n", " ").strip()
            continue  

        body_blocks.append((y0, x0, text, max_size))

    body_blocks.sort(key=lambda b: (round(b[0] / 4), b[1]))

    printed_page = None
    section_ref = None
    if header_text:
        digits = PAGE_NUM_RE.findall(header_text)
        if digits:
            printed_page = int(digits[0])
        section_ref = PAGE_NUM_RE.sub("", header_text).strip(" .") or None

    md_lines: list[str] = []
    current_heading = None
    for _, _, text, size in body_blocks:
        heading = _heading_level(text) if size >= HEADING_SIZE_THRESHOLD else None
        if heading:
            level, heading_text = heading
            current_heading = heading_text
            md_lines.append(f"{'#' * min(level + 1, 6)} {heading_text}")
        else:
            md_lines.append(text)

    body_text = "\n".join(md_lines).strip()

    return {
        "printed_page": printed_page,
        "section_ref": section_ref,
        "heading": current_heading,
        "text": body_text,
    }


In [48]:
doc = fitz.open(pdf_path)
n_pages = min(FIRST_N_PAGES, doc.page_count)

jsonl_path = out_dir / "pages.jsonl"
md_path = out_dir / "verifikacija_softvera.md"

with jsonl_path.open("w", encoding="utf-8") as jf, md_path.open("w", encoding="utf-8") as mf:
    mf.write("# Verifikacija softvera (elektronska verzija, 2026)\n\n")
    for i in range(n_pages):
        pdf_page = i + 1
        record = extract_page(doc[i])
        record["pdf_page"] = pdf_page

        (pages_dir / f"page_{pdf_page:04d}.txt").write_text(record["text"], encoding="utf-8")
        jf.write(json.dumps(record, ensure_ascii=False) + "\n")

        if record["text"]:
            page_label = record["printed_page"] if record["printed_page"] is not None else pdf_page
            mf.write(f"\n<!-- pdf_page={pdf_page} printed_page={page_label} -->\n\n")
            mf.write(record["text"] + "\n")

print(f"Ekstrahovano {n_pages} strana -> {pages_dir}, {jsonl_path}, {md_path}")


Ekstrahovano 204 strana -> /home/anja/Desktop/MU/Student-Question-Answering-from-Course-Materials/data/processed/pages, /home/anja/Desktop/MU/Student-Question-Answering-from-Course-Materials/data/processed/pages.jsonl, /home/anja/Desktop/MU/Student-Question-Answering-from-Course-Materials/data/processed/verifikacija_softvera.md


In [50]:
records = [json.loads(line) for line in jsonl_path.read_text(encoding="utf-8").splitlines() if line]
non_empty = [r for r in records if r["text"]]
print(f"Ukupno strana: {len(records)}, sa tekstom: {len(non_empty)}")

sample = next(r for r in records if r["pdf_page"] == 56)
print("--- primer strane 100 ---")
print(sample["text"][:600])

Ukupno strana: 204, sa tekstom: 198
--- primer strane 100 ---
jeziku, čime se eliminiše dvosmislenost i omogućava proverljiva interpretacĳa. Formalna semantika predstavlja temelj za razumevanje i analiziranje značenja softverskog koda na precizan, matematički način.
Upravo kroz formalnu semantiku postaje moguće:
-modelovati izvršavanje programa nezavisno od konkretne mašine ili kompajlera,
-dokazivati korektnost softverskih komponenti i -razvĳati alate za formalnu verifikacĳu.
Idealno rešenje za verifikacĳu softvera bi bio alat koji automatski analizira kôd i daje precizne informacĳe o njegovoj ispravnosti. Međutim, postoji fundamentalno ograničenje zbog
